In [1]:
from configuracoes_notebooks import set_proj_dir
set_proj_dir()

O diretorio do seu projeto é coleta_cebrap
Caminho absoluto do diretorio encontrado C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap
Caminho no path.


In [2]:
import geopandas as gpd
import os

from notebooks.jupyter import utils
from utils import (
    get_data_diretorio
)

# Ocorrências de inundações por distrito
Com base nas intersecções que temos, podemos somar os pontas para sabermos o total em cada distrito

In [3]:
data_path = get_data_diretorio()
assets_path = os.path.join(
    data_path,
    'assets'
)

# Dependências
Este notebook é dependente dos arquivos resultantes dos notebooks "overlay_pontos_inundacao_2024" e "../../arborizacao_viaria/malha_distritos"

In [4]:
gdf_distritos = gpd.read_parquet(
    os.path.join(
        assets_path,
        'distrito_ibge.parquet'
    )
)

In [5]:
overlay_inund = gpd.read_parquet(
    os.path.join(
        assets_path,
        'areas_inundacao',
        'interseccao_inundacoes2024_distritos.parquet'
    )
)

# Soma de pontos por distrito

In [6]:
for i, distrito in enumerate(gdf_distritos['CD_DIST']):
    gdf_distritos.loc[gdf_distritos['CD_DIST']==distrito, 'qt_inund_ocor'] = (
        len(
            overlay_inund
            .loc[overlay_inund['CD_DIST'] == distrito]
        )
    )

In [7]:
gdf_distritos.sample(2)

,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,qt_inund_ocor
77,3550308,São Paulo,355030878,SE,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,2.180669,23832,13885,"POLYGON ((333930.933 7394149.741, 333884.447 7...",0.0
21,3550308,São Paulo,355030822,CIDADE ADEMAR,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,12.233190,249218,103713,"POLYGON ((333467.739 7379561.767, 333458.192 7...",0.0


# Conferir

Distritos sem pontos de ocorrências de inundações:

In [8]:
zero_dists=(
    gdf_distritos
    .loc[gdf_distritos['qt_inund_ocor']==float(0)]
)
zero_dists.shape

(38, 15)

In [9]:
zero_dists.sample(2)

,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,qt_inund_ocor
51,3550308,São Paulo,355030852,MARSILAC,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,207.501260,11451,6139,"POLYGON ((333016.008 7354600.919, 333056.044 7...",0.0
15,3550308,São Paulo,355030816,CAMPO GRANDE,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,12.991325,115925,50009,"POLYGON ((329675.249 7379393.452, 329715.641 7...",0.0


Distrito com o maior número de ocorrências de inundações:

In [10]:
(
    gdf_distritos
    .loc[gdf_distritos['qt_inund_ocor']==gdf_distritos['qt_inund_ocor'].max()]
)

,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,qt_inund_ocor
36,3550308,São Paulo,355030837,ITAQUERA,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,14.699172,210960,85467,"POLYGON ((351668.035 7394259.261, 351646.256 7...",18.0


Distrito com o menor número de inundações (excluindo os sem ocorrência nenhuma):

In [11]:
min_positiv=(
    gdf_distritos
    .loc[gdf_distritos['qt_inund_ocor']>0]
)
min_dists=(
    gdf_distritos
    .loc[gdf_distritos['qt_inund_ocor']==min_positiv['qt_inund_ocor'].min()]
)
min_dists.shape

(23, 15)

In [12]:
min_dists.sample(2)

,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,qt_inund_ocor
56,3550308,São Paulo,355030857,PARQUE DO CARMO,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,15.650945,74677,30378,"POLYGON ((351246.547 7389761.772, 351233.56 73...",1.0
78,3550308,São Paulo,355030879,SOCORRO,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,12.261579,38051,16549,"POLYGON ((326832.519 7378799.896, 326814.871 7...",1.0


Nós vimos que a subprefeitura com mais pontos de ocorrências de inundação é a de São Miguel Paulista. Vamos confirmar se mantém quando calculamos pelos distritos:

In [13]:
overlay_inund.columns

Index(['data', 'ocorrencia', 'nm_subpref', 'id_ocorrencia', 'CD_MUN', 'NM_MUN',
       'CD_DIST', 'NM_DIST', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI',
       'CD_CONCURB', 'NM_CONCURB', 'AREA_KM2', 'total_pop', 'total_dom',
       'geometry'],
      dtype='object')

In [14]:
subpref_confere = gdf_distritos.copy()
for i, distrito in enumerate(subpref_confere['CD_DIST']):
    if overlay_inund['CD_DIST'].str.contains(distrito).any():
        subpref_confere.loc[subpref_confere['CD_DIST']==distrito, 'subpref']=(
            overlay_inund
            .loc[overlay_inund['CD_DIST']==distrito]
            .head(1)['nm_subpref']
            .values[0]
        )

In [15]:
subpref_confere.sample(5)

,CD_MUN,NM_MUN,CD_DIST,NM_DIST,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,AREA_KM2,total_pop,total_dom,geometry,qt_inund_ocor,subpref
7,3550308,São Paulo,355030808,BELEM,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,6.121560,55785,25449,"POLYGON ((337329.398 7395388.459, 337268.595 7...",0.0,NaN
18,3550308,São Paulo,355030819,CAPAO REDONDO,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,13.791801,270767,110144,"POLYGON ((318763.963 7379926.318, 318735.604 7...",5.0,CL - CAMPO LIMPO
5,3550308,São Paulo,355030806,BARRA FUNDA,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,5.884671,33436,16962,"POLYGON ((329146.056 7396609.887, 329100.802 7...",0.0,NaN
51,3550308,São Paulo,355030852,MARSILAC,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,207.501260,11451,6139,"POLYGON ((333016.008 7354600.919, 333056.044 7...",0.0,NaN
23,3550308,São Paulo,355030824,CIDADE LIDER,3501,São Paulo,350001,São Paulo,3550308,São Paulo/SP,10.558185,136660,54530,"POLYGON ((347490.062 7391594.735, 347468.66 73...",1.0,IQ - ITAQUERA
